# Environment and GPU setup

In [1]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.7.1+cu128
CUDA available: True
Selected device: cuda
GPU: NVIDIA GeForce GTX 1650
CUDA version: 12.8


# English–French sentence pairs

In [2]:
pairs = [
    ("hello", "vanakkam"),
    ("thank you", "nandri"),
    ("good morning", "kaalai vanakkam"),
    ("good night", "iruavu vanakkam"),
    ("i am happy", "naan magilchiyaga iruken"),
    ("i am sad", "naan sogama iruken"),
    ("how are you", "nee epdi iruka"),
    ("i love you", "naan unnai kaathalikiren"),
    ("see you soon", "seekram unnaya paakren"),
    ("goodbye", "poitu varen"),
]

# Building vocuabulary 

In [3]:
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

def build_vocab(sentences):

    words = set()

    for sentence in sentences:
        words.update(sentence.split())

    tokens = [PAD,SOS,EOS,UNK] + sorted(words)

    word_to_id = {word:id for id,word in enumerate(tokens)}
    id_to_word = {id:word for word,id in word_to_id.items()}

    return word_to_id,id_to_word

source_sentences = [source for source,target in pairs]
target_sentences = [target for source,target in pairs]

source_vocab,source_id_to_word = build_vocab(source_sentences)
target_vocab,target_id_to_word = build_vocab(target_sentences)

example = "i am happy _____ ".split()

example_id = [
    source_vocab.get(word,source_vocab[UNK])
    for word in example
]

print(example)
print(example_id)

['i', 'am', 'happy', '_____']
[11, 4, 8, 3]


# Encode the Sentence : adding SOS and EOS

In [4]:
def encode_sentence(sentence,vocab):

    token_id = [
        vocab.get(word,vocab[UNK])
        for word in sentence.split()
    ]

    token_id = [vocab[SOS]] + token_id +[vocab[EOS]]

    return token_id

example = 'i am happy'

token_ids = encode_sentence(example,source_vocab)

print(token_ids)

print([source_id_to_word[token] for token in token_ids])

# encoding every source and target sentences 
source_sequences = [
    encode_sentence(source,source_vocab)
    for source in source_sentences
]

target_sequences = [
    encode_sentence(target,target_vocab)
    for target in target_sentences
]

# convert everything to tensor 
source_tensor = [
    torch.tensor(sequence,dtype = torch.long)
    for sequence in source_sequences
]

target_tensor = [
    torch.tensor(sequence,dtype = torch.long)
    for sequence in target_sequences
]

[1, 11, 4, 8, 2]
['<SOS>', 'i', 'am', 'happy', '<EOS>']


# Padding the Sequences 

In [5]:
from torch.nn.utils.rnn import pad_sequence

source_padding_id = source_vocab[PAD]
target_padding_id = target_vocab[PAD]

# Pad sequences to the longest sequence in their respective batch.
# batch_first=True gives shape: (batch_size, sequence_length).
source_batch = pad_sequence(
    source_tensor,
    batch_first= True,
    padding_value=source_padding_id
)

target_batch = pad_sequence(
    target_tensor,
    batch_first= True,
    padding_value=target_padding_id
)

print("Source And Target Shape: ",source_batch.shape,' ',target_batch.shape)
print(source_batch[:2])
print(target_batch[:2])

Source And Target Shape:  torch.Size([10, 5])   torch.Size([10, 5])
tensor([[ 1,  9,  2,  0,  0],
        [ 1, 18, 19,  2,  0]])
tensor([[ 1, 20,  2,  0,  0],
        [ 1, 12,  2,  0,  0]])


# Create the padding mask

In [6]:
source_mask = source_batch != source_padding_id
target_mask = target_batch != target_padding_id

print(source_mask[:2])
print(target_mask[:2])

print()

print("Source Real Tokens: ",source_mask.sum().item())
print("Source padding tokens:", (~source_mask).sum().item())

tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True, False]])
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True, False, False]])

Source Real Tokens:  43
Source padding tokens: 7


# Encoder

In [7]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        padding_id
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim= embedding_dim,
            padding_idx= padding_id
        )

        self.lstm = nn.LSTM(
            input_size= embedding_dim,
            hidden_size= hidden_dim,
            batch_first= True
        )

    def forward(self,source,source_lengths):

        embedded = self.embedding(source)

        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first= True,
            enforce_sorted= False
        )

        packed_outputs,(hidden,cell) = self.lstm(packed)

        outputs,_ = pad_packed_sequence(
            packed_outputs,
            batch_first= True,
            total_length= source.size(1)
        )

        return outputs,hidden,cell

EMBEDDING_DIM = 64
HIDDEN_DIM = 128

encoder = Encoder(
    vocab_size=len(source_vocab),
    embedding_dim= EMBEDDING_DIM,
    hidden_dim= HIDDEN_DIM,
    padding_id= source_padding_id
).to(device)    

In [8]:
source_lengths = source_mask.sum(dim = 1) 

source_batch_gpu = source_batch.to(device)

encoder.eval()
with torch.no_grad():
    enocder_outputs,hidden,cell = encoder(
        source_batch_gpu,
        source_lengths
    )

print(source_batch_gpu.shape)
print(enocder_outputs.shape)
print(hidden.shape)
print(cell.shape)
print(enocder_outputs.is_cuda)

torch.Size([10, 5])
torch.Size([10, 5, 128])
torch.Size([1, 10, 128])
torch.Size([1, 10, 128])
True


In [9]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        padding_id
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim= embedding_dim,
            padding_idx= padding_id
        )

        self.lstm = nn.LSTM(
            input_size = embedding_dim,
            hidden_size = hidden_dim,
            batch_first = True
        )

        self.output_layer = nn.Linear(
            hidden_dim,vocab_size
        )

    def forward(self,target_token,hidden,cell):

        embedded = self.embedding(target_token)

        output,(hidden,cell) = self.lstm(embedded,(hidden,cell))

        logits = self.output_layer(output)

        return logits,hidden,cell

In [10]:
decoder = Decoder(
    vocab_size=len(target_vocab),
    embedding_dim= EMBEDDING_DIM,
    hidden_dim= HIDDEN_DIM,
    padding_id= target_padding_id
).to(device)

target_token = torch.full(
    (source_batch.size(0), 1),
    fill_value=target_vocab[SOS],
    dtype=torch.long,
    device=device
)

decoder.eval()

with torch.no_grad():
    logits, decoder_hidden, decoder_cell = decoder(
        target_token,
        hidden,
        cell
    )

print("Target token shape:", target_token.shape)
print("Logits shape:", logits.shape)
print("Decoder hidden shape:", decoder_hidden.shape)
print("Decoder cell shape:", decoder_cell.shape)
print("Decoder on GPU:", logits.is_cuda)

Target token shape: torch.Size([10, 1])
Logits shape: torch.Size([10, 1, 22])
Decoder hidden shape: torch.Size([1, 10, 128])
Decoder cell shape: torch.Size([1, 10, 128])
Decoder on GPU: True


In [11]:
class seq2seq(nn.Module):

    def __init__(self,encoder,decoder,sos_id, pad_id):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.sos_id  = sos_id
        self.pad_id  = pad_id

    def forward(self,source,source_lengths,target):

        _,hidden,cell = self.encoder(source,source_lengths)

        target_input = target[:,:-1]
        all_logits = []

        for t in range(target_input.size(1)):
            token = target_input[:,t].unsqueeze(1) 

            logits,hidden,cell= self.decoder(token,hidden,cell)

            all_logits.append(logits.squeeze(1)) 

        return torch.stack(all_logits,dim =1)

In [12]:
model = seq2seq(
    encoder=encoder,
    decoder=decoder,
    sos_id=target_vocab[SOS],
    pad_id=target_padding_id
).to(device)

In [13]:
source_gpu = source_batch.to(device)
target_gpu = target_batch.to(device)

source_lengths = source_mask.sum(dim = 1).cpu()

model.eval()
with torch.no_grad():
    logits = model(source_gpu, source_lengths, target_gpu)

expected = target_gpu[:, 1:]

print("Logits shape:", logits.shape)
print("Expected shape:", expected.shape)
print("Same sequence dimensions:", logits.shape[:2] == expected.shape)

Logits shape: torch.Size([10, 4, 22])
Expected shape: torch.Size([10, 4])
Same sequence dimensions: True


In [14]:
criterion = nn.CrossEntropyLoss(ignore_index=target_padding_id)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

source_gpu = source_batch.to(device)
target_gpu = target_batch.to(device)

source_lengths = source_mask.sum(dim = 1).cpu()


EPOCHS = 200 

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    logits = model(source_gpu,source_lengths,target_gpu)

    expected = target_gpu[:,1:]

    loss = criterion(
        logits.reshape(-1,len(target_vocab)),
        expected.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {loss.item():.4f}")

Epoch [20/200], Loss: 0.0035
Epoch [40/200], Loss: 0.0003
Epoch [60/200], Loss: 0.0002
Epoch [80/200], Loss: 0.0002
Epoch [100/200], Loss: 0.0002
Epoch [120/200], Loss: 0.0001
Epoch [140/200], Loss: 0.0001
Epoch [160/200], Loss: 0.0001
Epoch [180/200], Loss: 0.0001
Epoch [200/200], Loss: 0.0001


In [15]:
def translate(sentence,max_length = 10):

    model.eval()

    source_ids = encode_sentence(sentence.lower(),source_vocab)

    source = torch.tensor(
        [source_ids],dtype = torch.long,device = device
    )

    source_lengths = torch.tensor([len(source_ids)])

    with torch.no_grad():
        # Encode the source sentence
        _, hidden, cell = model.encoder(source, source_lengths)

        # Begin generation with SOS
        token = torch.tensor(
            [[target_vocab[SOS]]], dtype=torch.long, device=device
        )

        predicted_ids = []

        for _ in range(max_length):
            # Predict the next token
            logits, hidden, cell = model.decoder(token, hidden, cell)

            # Select the token with the highest probability
            next_id = logits[:, -1, :].argmax(dim=-1).item()

            # Stop when EOS is generated
            if next_id == target_vocab[EOS]:
                break

            predicted_ids.append(next_id)

            # Feed the prediction back into the decoder
            token = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    # Convert predicted IDs back into words
    words = [
        target_id_to_word[idx]
        for idx in predicted_ids
        if idx not in (target_vocab[SOS], target_vocab[PAD])
    ]

    return " ".join(words)

In [16]:
# Test sentences from the training dataset
test_sentences = [
    "hello",
    "thank you",
    "good night",
    "i am happy",
    "i love you",
    "goodbye"
]

for sentence in test_sentences:
    print(f"English: {sentence}")
    print(f"French:  {translate(sentence)}")
    print("-" * 30)

English: hello
French:  vanakkam
------------------------------
English: thank you
French:  nandri
------------------------------
English: good night
French:  iruavu vanakkam
------------------------------
English: i am happy
French:  naan magilchiyaga iruken
------------------------------
English: i love you
French:  naan unnai kaathalikiren
------------------------------
English: goodbye
French:  poitu varen
------------------------------


In [19]:
print(translate("good morning"))

kaalai vanakkam
